# 🎨 FaceBlend — SDXL + IP-Adapter FaceID

**Gera imagens 1024×1024** com a identidade combinada de **duas pessoas**,  
rodando 100% na nuvem com GPU A100 (Google Colab Pro+ / AI Premium).

---

### ▶️ Como usar
1. `Runtime → Change runtime type → A100 GPU`
2. Execute as células **1 a 4** uma vez (instalação ~5 min)
3. Faça upload das suas fotos no Google Drive (veja célula 3)
4. Edite os **caminhos das fotos** e a **lista de prompts** na célula 5
5. Execute a célula 5 — as imagens serão salvas no Drive automaticamente

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CÉLULA 1 — Verificar GPU e clonar repositório       ║
# ╚══════════════════════════════════════════════════════╝
import subprocess, sys

# Verificar GPU disponível
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
print('🖥️  GPU detectada:')
print(result.stdout.strip() if result.returncode == 0 else '⚠️  Sem GPU! Vá em Runtime → Change runtime type → A100 GPU')

# Clonar o repositório (substitua pelo seu usuario/repo)
GITHUB_REPO = 'YOUR_GITHUB_USERNAME/sdxl-faceblend'  # ← edite aqui

!git clone https://github.com/{GITHUB_REPO}.git /content/sdxl-faceblend 2>&1 | tail -3
%cd /content/sdxl-faceblend
print('✓ Repositório clonado em /content/sdxl-faceblend')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CÉLULA 2 — Instalar dependências (~5 min)           ║
# ╚══════════════════════════════════════════════════════╝
# PyTorch já vem instalado no Colab; só instalar o resto
!pip install -q \
    diffusers>=0.25.0 \
    transformers>=4.37.0 \
    accelerate>=0.27.0 \
    safetensors>=0.4.0 \
    xformers \
    insightface>=0.7.3 \
    onnxruntime-gpu>=1.17.0 \
    opencv-python-headless>=4.9.0 \
    huggingface-hub>=0.21.0

# IP-Adapter do repo oficial Tencent
!pip install -q git+https://github.com/tencent-ailab/IP-Adapter.git

print('\n✓ Todas as dependências instaladas!')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CÉLULA 3 — Montar Google Drive                      ║
# ╚══════════════════════════════════════════════════════╝
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# Pastas no seu Drive
FACES_DIR  = '/content/drive/MyDrive/faceblend/faces'    # coloque suas fotos aqui
OUTPUT_DIR = '/content/drive/MyDrive/faceblend/output'   # imagens geradas

os.makedirs(FACES_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✓ Drive montado!')
print(f'📁 Coloque as fotos de rosto em:\n   {FACES_DIR}')
print(f'📁 As imagens geradas serão salvas em:\n   {OUTPUT_DIR}')

# Listar fotos já presentes
fotos = [f for f in os.listdir(FACES_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]
print(f'\nFotos encontradas: {fotos if fotos else "nenhuma ainda — faça upload no Drive"}')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CÉLULA 4 — Baixar pesos IP-Adapter FaceID SDXL      ║
# ║  (executa UMA vez; ~1.5 GB)                          ║
# ╚══════════════════════════════════════════════════════╝
from huggingface_hub import hf_hub_download
import os

IP_CKPT = '/content/sdxl-faceblend/ip_adapter_faceid_sdxl.bin'

if not os.path.exists(IP_CKPT):
    print('⬇️  Baixando ip_adapter_faceid_sdxl.bin (~1.5 GB)...')
    hf_hub_download(
        repo_id   = 'h94/IP-Adapter-FaceID',
        filename  = 'ip_adapter_faceid_sdxl.bin',
        local_dir = '/content/sdxl-faceblend',
    )
    print('✓ Download concluído!')
else:
    print('✓ Pesos já presentes, pulando download.')

print(f'   Arquivo: {IP_CKPT}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 5 — ✏️ EDITE AQUI: fotos, prompts e parâmetros             ║
# ╚══════════════════════════════════════════════════════════════════════╝
import sys
sys.path.insert(0, '/content/sdxl-faceblend')

from generate_faceblend import build_pipeline, build_face_analyzer, \
                               average_face_embeddings, run_generation_loop
import torch

# ─── 📸 Caminhos das duas fotos de rosto ─────────────────────────────────────
# As fotos devem estar em: /content/drive/MyDrive/faceblend/faces/
FACE_A = f'{FACES_DIR}/pessoa_a.jpg'   # ← troque pelo nome do arquivo
FACE_B = f'{FACES_DIR}/pessoa_b.jpg'   # ← troque pelo nome do arquivo

# ─── 📝 Lista de prompts ─────────────────────────────────────────────────────
PROMPTS = [
    "a person in a futuristic space station, cinematic lighting, sharp focus, 8k",
    "a Renaissance oil painting portrait, museum quality, dramatic lighting",
    "a person hiking in Patagonia at golden hour, wide angle, landscape photo",
]

# ─── ⚙️ Parâmetros de geração ────────────────────────────────────────────────
NUM_STEPS  = 40      # inference steps
CFG_SCALE  = 7.5     # guidance scale
SEED       = 42      # semente inicial
IP_SCALE   = 0.8     # força do IP-Adapter (0.0–1.0)

# ─────────────────────────────────────────────────────────────────────────────
# A partir daqui é automático — não precisa editar
# ─────────────────────────────────────────────────────────────────────────────

print('🔧 Inicializando pipeline SDXL + IP-Adapter FaceID...')
ip_model = build_pipeline(ip_ckpt=IP_CKPT)

print('\n🧠 Extraindo e combinando embeddings faciais...')
analyzer    = build_face_analyzer()
face_embeds = average_face_embeddings(analyzer, FACE_A, FACE_B)
face_embeds = face_embeds.to('cuda', dtype=torch.float16)

print(f'\n🎨 Gerando {len(PROMPTS)} imagem(ns)...')
saved = run_generation_loop(
    ip_model    = ip_model,
    face_embeds = face_embeds,
    prompts     = PROMPTS,
    num_steps   = NUM_STEPS,
    cfg_scale   = CFG_SCALE,
    seed        = SEED,
    output_dir  = OUTPUT_DIR,
    scale       = IP_SCALE,
)

# Exibir inline no notebook
from IPython.display import display
from PIL import Image

print('\n🖼️  Resultado:')
for path in saved:
    print(f'   {path}')
    display(Image.open(path).resize((512, 512)))